Carga de librerías y definir ruta

In [11]:
import pandas as pd
import geopandas as gpd
import os
import re

# Localizador de rutas relativo (Funciona en local y en carpetas de notebooks)
if os.path.exists('data'):
    PATH_DATA = 'data'
elif os.path.exists(os.path.join('..', 'data')):
    PATH_DATA = os.path.join('..', 'data')
else:
    PATH_DATA = 'data'

print(f"📂 Directorio de datos configurado en: {os.path.abspath(PATH_DATA)}")

📂 Directorio de datos configurado en: /Users/curro/Trabajo IA/Trabajo-Inteligencia-Artificial/data


ETL espacial

In [12]:
ruta_shp_espana = os.path.join(PATH_DATA, 'SECC_CE_20230101.shp')
ruta_geojson_sevilla = os.path.join(PATH_DATA, 'mapa_sevilla.geojson')

if os.path.exists(ruta_shp_espana):
    mapa_espana = gpd.read_file(ruta_shp_espana)
    mapa_sevilla = mapa_espana[mapa_espana['CUSEC'].astype(str).str.startswith('41091')].copy()
    mapa_sevilla.to_file(ruta_geojson_sevilla, driver='GeoJSON')
    print(f" Mapa creado con {len(mapa_sevilla)} secciones.")
else:
    print(f"Error: No se encuentra {ruta_shp_espana}")

 Mapa creado con 521 secciones.


ETL del archivo listings.csv

In [2]:
  # Definición de columnas necesarias
final_cols = [
      'id', 'host_id', 'host_is_superhost', 'neighbourhood_cleansed', 
      'neighbourhood_group_cleansed', 'latitude', 'longitude', 
      'property_type', 'room_type', 'accommodates', 'bathrooms_text', 
      'bedrooms', 'price', 'minimum_nights', 'number_of_reviews', 
      'review_scores_rating', 'license', 'instant_bookable',
      'availability_365', 'calculated_host_listings_count', 
      'reviews_per_month', 'amenities'
  ]

ruta_listings = os.path.join(PATH_DATA, 'listings.csv')

if os.path.exists(ruta_listings):
      df_listings = pd.read_csv(ruta_listings, usecols=final_cols, low_memory=False)
      
      # Limpieza de precio: de "$1,200.00" a 1200.0
      df_listings['price'] = df_listings['price'].replace(r'[\$,]', '', regex=True).astype(float)
      
      print(f"✅ Listings procesados. Dimensiones: {df_listings.shape}")
else:
      print(f"❌ Error: No se encuentra listings.csv")

✅ Listings procesados. Dimensiones: (8215, 22)


ETL del archivo calendar.csv

In [14]:
cols_calendar = ['listing_id', 'date', 'available', 'minimum_nights', 'maximum_nights']
ruta_calendar = os.path.join(PATH_DATA, 'calendar.csv.zip')

if os.path.exists(ruta_calendar):
    df_calendar = pd.read_csv(ruta_calendar, usecols=cols_calendar, low_memory=False, compression='zip')
    
    df_calendar['date'] = pd.to_datetime(df_calendar['date'])
    df_calendar['available'] = df_calendar['available'].map({'t': True, 'f': False})
    df_calendar = df_calendar.sort_values(by=['listing_id', 'date'])
    
    # Guardamos por separado ya que es un volumen de datos distinto
    df_calendar.to_csv(os.path.join(PATH_DATA, 'calendar_limpio.csv.zip'), index=False, compression='zip')
    print(f"✅ Calendario procesado y guardado. Dimensiones: {df_calendar.shape}")

✅ Calendario procesado y guardado. Dimensiones: (2998475, 5)


ETL del archivo de rentas

In [ ]:
ruta_renta_raw = os.path.join(PATH_DATA, 'renta_provincia_sevilla.csv')

if os.path.exists(ruta_renta_raw):
    df_renta_raw = pd.read_csv(ruta_renta_raw, sep=';', encoding='latin-1', decimal=',')
    
    # Filtrado: Sevilla Capital (Código 41091) y limpieza
    df_renta = df_renta_raw[
        (df_renta_raw['Secciones'].notna()) & 
        (df_renta_raw['Secciones'].astype(str).str.contains('^41091'))
    ].copy()
    
    if df_renta['Total'].dtype == 'object':
        df_renta['Total'] = df_renta['Total'].str.replace('.', '', regex=False).astype(float)
    
    df_renta.to_csv(os.path.join(PATH_DATA, 'renta_sevilla_capital_limpio.csv'), index=False)
    print(f"✅ Renta de Sevilla Capital lista. Secciones: {len(df_renta)}")

✅ Renta de Sevilla Capital lista. Secciones: 541


Exportación final

In [18]:
# 1. Convertir Airbnb a Geodataframe
df_renta['renta_media'] = df_renta['Total'].str.replace('.', '', regex=False).astype(float)

gdf_listings = gpd.GeoDataFrame(
    df_listings, 
    geometry=gpd.points_from_xy(df_listings.longitude, df_listings.latitude), 
    crs="EPSG:4326"
).to_crs(mapa_sevilla.crs)

# 2. Join Espacial  
pisos_con_seccion = gpd.sjoin(gdf_listings, mapa_sevilla[['CUSEC', 'geometry']], how="left", predicate="intersects")

# 3. Merge con Renta
df_renta['Secciones'] = df_renta['Secciones'].astype(str)
pisos_con_seccion['CUSEC'] = pisos_con_seccion['CUSEC'].astype(str)

dataset_final = pd.merge(pisos_con_seccion, df_renta[['Secciones', 'renta_media']], 
                        left_on='CUSEC', right_on='Secciones', how='left')

# 4. Guardar dataset
dataset_final.to_csv(os.path.join(PATH_DATA, 'dataset_final.csv'), index=False)
print("🏁 ¡HECHO! Tienes el dataset definitivo en 'dataset_final.csv'")

🏁 ¡HECHO! Tienes el dataset definitivo en 'dataset_final.csv'


Guardado de los dataframes limpios para su posterior uso